!!!  Data Quality Check — Indian E-Commerce Dataset

Checked the raw data before building the SQL schema. Dataset: 40,000 customers,
2,000 products, 250,000 orders (June 2024 – June 2026).

## What I found

- No duplicate IDs in any table.
- No broken links — every order matches a real customer and product.
- Nulls in `sales` are normal, not errors: ~80% of orders had no coupon,
  ~52% weren't reviewed. `customers` and `products` have no nulls at all.
- `Delivery_Date` is an *expected* delivery date, not the actual one —
  even Cancelled/Processing orders have a delivery date ~4.5 days out,
  which only makes sense if it's a planned date.
- **Big finding:** 99% of orders still stuck in "Processing" have been there
  longer than a normal delivery should take (7+ days). Real bottleneck, not noise.

## Conclusion
Data is clean and reliable. Moving on to SQL schema + analysis.

In [ ]:
import pandas as pd 


In [2]:
customers = pd.read_csv('../data/raw/customers.csv')
products = pd.read_csv('../data/raw/products.csv')
sales = pd.read_csv('../data/raw/sales.csv')


In [3]:
print("Customers:", customers.shape)
print("Products:", products.shape)
print("Sales:", sales.shape)

Customers: (40000, 15)
Products: (2000, 12)
Sales: (250000, 21)


In [5]:
print("Nulls in customers")
print(customers.isna().sum()[customers.isna().sum() > 0])

Nulls in customers
Series([], dtype: int64)


In [6]:
print(" Nulls in products ")
print(products.isna().sum()[products.isna().sum() > 0])

 Nulls in products 
Series([], dtype: int64)


In [7]:
print("Nulls in sales ")
print(sales.isna().sum()[sales.isna().sum() > 0])

Nulls in sales 
Coupon_Code    199815
Rating         129970
Review_Text    129970
dtype: int64


In [8]:
print("Duplicate Customer_ID:", customers['Customer_ID'].duplicated().sum())
print("Duplicate Product_ID:", products['Product_ID'].duplicated().sum())
print("Duplicate Order_ID:", sales['Order_ID'].duplicated().sum())

Duplicate Customer_ID: 0
Duplicate Product_ID: 0
Duplicate Order_ID: 0


In [9]:
missing_cust = ~sales['Customer_ID'].isin(customers['Customer_ID'])
missing_prod = ~sales['Product_ID'].isin(products['Product_ID'])

print("Sales with missing Customer_ID:", missing_cust.sum())
print("Sales with missing Product_ID:", missing_prod.sum())


Sales with missing Customer_ID: 0
Sales with missing Product_ID: 0


In [10]:
sales['Order_Date'] = pd.to_datetime(sales['Order_Date'])
sales['Delivery_Date'] = pd.to_datetime(sales['Delivery_Date'])
sales['Delivery_Days'] = (sales['Delivery_Date'] - sales['Order_Date']).dt.days

print("Delivery before Order (impossible):", (sales['Delivery_Date'] < sales['Order_Date']).sum())
print()
print("Avg delivery days by status:")
print(sales.groupby('Order_Status')['Delivery_Days'].mean())

Delivery before Order (impossible): 0

Avg delivery days by status:
Order_Status
Cancelled     4.477413
Delivered     4.500892
Processing    4.517013
Returned      4.510526
Shipped       4.496268
Name: Delivery_Days, dtype: float64


In [11]:
max_date = sales['Order_Date'].max()
cutoff = max_date - pd.Timedelta(days=7)

stuck = sales[(sales['Order_Status'] == 'Processing') & (sales['Order_Date'] < cutoff)]
total_processing = (sales['Order_Status'] == 'Processing').sum()

print("Latest order date in dataset:", max_date)
print("Total Processing orders:", total_processing)
print("Processing orders older than 7 days:", len(stuck))
print("Percent stuck:", round(len(stuck) / total_processing * 100, 1), "%")

Latest order date in dataset: 2026-06-30 00:00:00
Total Processing orders: 12402
Processing orders older than 7 days: 12276
Percent stuck: 99.0 %
